# Phase 5: Hybrid Recommender System Design

## 🎯 Objective
To build an intelligent recommendation engine that suggests professors based on two factors:
1.  **Semantic Relevance:** How well the professor's reviews match the user's search query (using **TF-IDF** & **Cosine Similarity**).
2.  **Statistical Quality:** The professor's overall reputation, adjusted for the number of reviews using **Bayesian Average** to prevent bias from low-volume data.

## 🛠 Architecture
The system follows a **Weighted Hybrid** approach:
$$Final Score = (w_1 \times Semantic Score) + (w_2 \times Bayesian Rating)$$
*Where $w_1 = 0.7$ (Focus on relevance) and $w_2 = 0.3$ (Focus on quality).*

In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics.pairwise import cosine_similarity

# Load processed data
reviews_df = pd.read_csv('../data/processed/cleaned_reviews.csv')
profiles_df = pd.read_csv('../data/processed/professor_profiles.csv')

# Load the pre-trained TF-IDF Vectorizer (from phase 3)
# This ensures we use the exact same vocabulary as the training phase
vectorizer = joblib.load('../models/tfidf_vectorizer.pkl')

print(f"✅ Data Loaded. Reviews: {len(reviews_df)}, Profiles: {len(profiles_df)}")

✅ Data Loaded. Reviews: 4548, Profiles: 1427


In [2]:
# --- Step 1: Handle Missing Data (Crucial Fix) ---
# Ensure all comments are strings. This fixes the 'float found' TypeError.
reviews_df['comment_text'] = reviews_df['comment_text'].fillna('').astype(str)

# --- Step 2: Create Professor Corpus ---
# Concatenate all reviews for each professor into a single large document
corpus_df = reviews_df.groupby('professor_name_raw')['comment_text'].apply(lambda x: ' '.join(x)).reset_index()

# Merge with profile stats (Cluster, Scores)
rec_db = pd.merge(profiles_df, corpus_df, on='professor_name_raw', how='left')

# Handle any NaN created after merge
rec_db['comment_text'] = rec_db['comment_text'].fillna('')

print("✅ Professor Corpus Created successfully.")
rec_db.head(2)

✅ Professor Corpus Created successfully.


,professor_name_raw,avg_sentiment,review_count,avg_numeric_score,log_review_count,Cluster_Label,Cluster_Agg,PCA1,PCA2,comment_text
0,آبت قره قانی,0.695066,1,3.475330,0.693147,1,1,1.410312,-0.456472,سخت گیرانه نمره میدن\n~~~~~~~~~~~~~~~~~\nبرای ...
1,آتنا بروغنی,0.754962,1,3.774808,0.693147,1,1,1.763117,-0.447008,استاد خیلی خوبی هستن \nگروه بندی میکنن و از هر...


## 5.1 Bayesian Weighted Rating
We use the IMDB weighted rating formula to ensure fairness. A professor with a single 5-star review shouldn't rank higher than one with fifty 4.8-star reviews.

$$WR = (\frac{v}{v+m} \cdot R) + (\frac{m}{v+m} \cdot C)$$

* **v**: number of reviews
* **m**: minimum votes required
* **R**: average sentiment of the professor
* **C**: mean sentiment across the whole university

In [3]:
# Calculate C (Mean vote across whole dataset)
C = rec_db['avg_sentiment'].mean()

# Calculate m (Minimum votes required to be listed, e.g., 25th percentile)
m = rec_db['review_count'].quantile(0.25)

def weighted_rating(x, m=m, C=C):
    v = x['review_count']
    R = x['avg_sentiment']
    # Bayesian Formula
    return (v/(v+m) * R) + (m/(v+m) * C)

# Apply score
rec_db['bayesian_score'] = rec_db.apply(weighted_rating, axis=1)

print("✅ Bayesian Scores Calculated.")

✅ Bayesian Scores Calculated.


In [4]:
# Transform the entire professor corpus into numeric vectors
# This matrix represents the "Knowledge Base" of our system
tfidf_matrix = vectorizer.transform(rec_db['comment_text'])

print(f"✅ Search Engine Ready. Matrix Shape: {tfidf_matrix.shape}")

✅ Search Engine Ready. Matrix Shape: (1427, 6744)


In [5]:
def extract_snippet(text, query, window=100):
    """
    Finds the most relevant sentence in the text containing query keywords.
    Used for 'Explainability' in the dashboard.
    """
    if not text: return "No comments available."
    
    # Simple split by query words
    query_words = [w for w in query.split() if len(w) > 2]
    
    # Try to find the exact sentence
    sentences = text.split('.')
    for sent in sentences:
        if any(w in sent for w in query_words):
            return sent[:window] + "..."
            
    return text[:window] + "..."

def hybrid_recommender(query, top_n=5, weight_semantic=0.7, weight_quality=0.3):
    """
    Core Recommendation Logic.
    Args:
        query (str): User's search text (e.g., "khosh akhlagh")
        weight_semantic (float): Importance of text match (0.0 - 1.0)
        weight_quality (float): Importance of overall rating (0.0 - 1.0)
    """
    
    # 1. Vectorize User Query
    query_vec = vectorizer.transform([query])
    
    # 2. Semantic Search (Cosine Similarity)
    similarity = cosine_similarity(query_vec, tfidf_matrix).flatten()
    
    # 3. Create Temporary Result Frame
    results = rec_db.copy()
    results['semantic_score'] = similarity
    
    # 4. Normalize Scores (Min-Max Scaling for fair combination)
    # We normalize semantic score to match the scale of bayesian score (roughly 0-1)
    results['final_score'] = (results['semantic_score'] * weight_semantic) + \
                             (results['bayesian_score'] * weight_quality)
    
    # 5. Filter & Sort
    # Only return results with at least some semantic relevance (>0)
    final_results = results[results['semantic_score'] > 0].sort_values(by='final_score', ascending=False).head(top_n)
    
    # 6. Formatting Output
    output = []
    for _, row in final_results.iterrows():
        snippet = extract_snippet(row['comment_text'], query)
        output.append({
            "Professor": row['professor_name_raw'],
            "Match Score": f"{row['final_score']:.2f}",
            "Cluster": row['Cluster_Label'], # e.g., 'Top Rated', 'Strict'
            "Why?": snippet
        })
        
    return pd.DataFrame(output)

In [6]:
# --- Test Scenario 1: Personality ---
print("🔎 User asks: 'استاد خوش اخلاق و با سواد'")
display(hybrid_recommender("خوش اخلاق با سواد"))

# --- Test Scenario 2: Grading Policy ---
print("\n🔎 User asks: 'نمره خوب میده'")
display(hybrid_recommender("نمره خوب پاس شدن"))

# --- Test Scenario 3: Specific Complaint/Feature ---
print("\n🔎 User asks: 'سخت گیر ولی یاد میگیریم'")
display(hybrid_recommender("سخت گیر درس یاد میده"))

🔎 User asks: 'استاد خوش اخلاق و با سواد'


,Professor,Match Score,Cluster,Why?
0,استاد حمیدرضا جهانگیری,0.39,1,استاد بسیار خوش اخلاق و با دانش هستن ، کلاسشون...
1,ابریشمی,0.36,1,یکی از با سواد ترین استادای دانشگاه هستند \nو ...
2,امین اوحدی اصفهانی,0.34,1,استاد اخلاق بسیار خوبی دارند و جزوه خودشونو می...
3,محمدرضا عابدی,0.31,1,کلن استاد خوبیه ، هر کی تونست باهاش برداره الا...
4,دکتر باقر محمدصادقی,0.30,1,استاد خوش اخلاق و با سوادی هستند، برای درس از ...



🔎 User asks: 'نمره خوب میده'


,Professor,Match Score,Cluster,Why?
0,مسعود ذوالفقاری,0.33,1,استاد خوبین تدریس کاملی دارن و همه نکات رو میگ...
1,پورسید آقایی,0.32,2,استاد خیلی خوبی هستن\nبا دانش عمومی بالا\nحضور...
2,استاد شایگان منش,0.30,1,چون زیاد اطلاعات راجبشون نیست تصمیم گرفتم بنوی...
3,استاد امین عبداله زاده,0.30,1,دست آخر هم نمره خوب و با ارفاق میدن\n~~~~~~~~...
4,استاد سیدین و ابوطالبی,0.30,1,استاد سیدین خوش نمره نیست امتحاناش هم سخته ولی...



🔎 User asks: 'سخت گیر ولی یاد میگیریم'


,Professor,Match Score,Cluster,Why?
0,سمیه علم الهدی,0.36,1,استاد خوب و خوش اخلاقی هستند \nمیشه ازشون درس ...
1,علوی,0.33,0,خیلی سخت گیر هستند سرکلاس و نفس کشیدن سخت میشه...
2,استاد سیده محبوبه مولوی عربشاهی,0.30,1,استاد خوب و با سوادی هستن فقط خیلی سخت گیری شو...
3,استاد شریعتی,0.29,1,استاد خوبی هستند و آداب و معاشرت خوبی دارند و ...
4,احمد فروغ,0.28,0,در مجموع سخت گیر نیست و در زمینه تدریس هم حرفی...


In [9]:
# Save Database for Dashboard
rec_db.to_csv('../data/processed/recommendation_db.csv', index=False)
joblib.dump(tfidf_matrix, '../models/tfidf_search_matrix.pkl')
print("✅ Recommendation database and matrix saved for Dashboard.")

✅ Recommendation database and matrix saved for Dashboard.
